In [1]:
import glob

from tqdm.auto import tqdm

In [2]:
import dask.array as da
import tifffile
import zarr
from dask.diagnostics import ProgressBar
from ome_zarr.io import parse_url
from ome_zarr.writer import (
    add_metadata,
    write_image,
    write_multiscales_metadata,
)

In [3]:
tif_fns = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse*/tif/*')

In [7]:
tif_fns = [fn for fn in tif_fns if 'ome' not in fn]
print(tif_fns)

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/tif/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.tif', '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/tif/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.tif', '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_9/tif/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550.tif']


In [8]:
for tif_fn in tqdm(tif_fns):
    try:
        if 'ome' in tif_fn:
            print('skipping suspect corrupt')
            continue
        else:
            zarr_image = tifffile.imread(tif_fn, aszarr=True)
            dask_image = da.from_zarr(zarr_image)
            print(output_fn, dask_image.shape)
            dask_image = dask_image.transpose(1, 0, 2, 3,)  # lazy; no data copy
            # v0.5 / Zarr v3 store
            out_zarr = tif_fn.replace('.tif', '.zarr')
            store = parse_url(out_zarr, mode="w").store
            root = zarr.group(store=store)
            
            # writes data + builds a 2x YX pyramid by default
            with ProgressBar():  # optional live progress
                write_image(
                    image=dask_image,
                    group=root,
                    axes="czyx",
                    storage_options=dict(chunks=(1, 1, 512, 512)),  # (C,Z,Y,X)
                )
            
            # optional: channel labels for nicer viewing in napari/viv
            add_metadata(root, {"omero": {
                "channels": [
                    {"label": "CF405"},
                    {"label": "CF488"},
                    {"label": "CF561"},
                ]
            }})
        
            # after write_image(... axes="czyx")
            level_names = sorted(root.array_keys(), key=int)   # <-- not group_keys()
            
            axes = [
                {"name": "c", "type": "channel"},
                {"name": "z", "type": "space", "unit": "micrometer"},
                {"name": "y", "type": "space", "unit": "micrometer"},
                {"name": "x", "type": "space", "unit": "micrometer"},
            ]
            
            px_z, px_y, px_x = 2.0, 0.1625, 0.1625
            datasets = []
            for i, p in enumerate(level_names):
                datasets.append({
                    "path": p,
                    "coordinateTransformations": [
                        {"type": "scale", "scale": [1.0, px_z, px_y*(2**i), px_x*(2**i)]},  # C Z Y X
                        {"type": "translation", "translation": [0, 0, 0, 0]},
                    ]
                })
            
            write_multiscales_metadata(root, datasets=datasets, axes=axes)
    except:
        print(tif_fn, 'failed')

  0%|          | 0/3 [00:00<?, ?it/s]

<tifffile.TiffPages @16> invalid page offset 93957312470


/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/tif/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.tif failed


<tifffile.TiffPages @16> invalid page offset 89049585582
<tifffile.TiffPages @16> invalid page offset 3015743627


/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/tif/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.tif failed
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_9/tif/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550.tif failed


In [11]:
tif_fn = '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/tif/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.tif'
zarr_image = tifffile.imread(tif_fn, aszarr=True)

<tifffile.TiffPages @16> invalid page offset 93957312470


RuntimeError: incompatible keyframe

Traceback (most recent call last):
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_qt/widgets/qt_welcome.py", line 194, in dropEvent
    self.parent().parent().parent().dropEvent(event)
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_qt/qt_viewer.py", line 1271, in dropEvent
    self._open_from_list_of_urls_data(
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_qt/qt_viewer.py", line 1288, in _open_from_list_of_urls_data
    self._qt_open(
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_qt/qt_viewer.py", line 1106, in _qt_open
    self.viewer.open(
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/components/viewer_model.py", line 1405, in open
    layers = self._open_or_raise_error(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/components/viewer_model.py", lin

In [9]:
import napari

In [12]:
viewer = napari.Viewer(title = 'test loading zarr')

Traceback (most recent call last):
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/components/viewer_model.py", line 1505, in _open_or_raise_error
    added = self._add_layers_with_plugins(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/components/viewer_model.py", line 1596, in _add_layers_with_plugins
    layer_data, hookimpl = read_data_with_plugins(
                           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/plugins/io.py", line 78, in read_data_with_plugins
    res = _npe2.read(paths, plugin, stack=stack)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/plugins/_npe2.py", line 56, in read
    layer_data, reader = io_utils.read_get_reader(
                         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dayn/miniconda3/envs/godspee